# rebalancing_flags table

In [ ]:

import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests

from pathlib import Path

In [ ]:
cwd = Path.cwd()
data_dir = Path(cwd / 'data')

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

### stations that are empty more than 50% of the time during rush hour

In [ ]:
cursor.execute('''
    SELECT station_id,
        COUNT(*) AS rush_hour_snapshots,
      100 * SUM(CASE WHEN num_bikes_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_empty,
      100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_full
  FROM availability_snapshots
  WHERE EXTRACT(HOUR FROM captured_at) IN (7,8,9,17,18,19)
  GROUP BY station_id
  HAVING (100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*)) > 50
  ORDER BY pct_empty DESC;
''')
cursor.fetchall()

### graph the distribution

In [ ]:
query = '''
    SELECT station_id,
        COUNT(*) AS rush_hour_snapshots,
        100 * SUM(CASE WHEN num_bikes_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_empty
  FROM availability_snapshots
  WHERE EXTRACT(HOUR FROM captured_at) IN (7,8,9,17,18,19)
  GROUP BY station_id
  
  '''
df_empty = pd.read_sql(query, conn)

In [ ]:
df_empty.shape

In [ ]:
#graphing a histogram
plt.figure(figsize=(10,6))
plt.hist(df_empty['pct_empty'], bins = 30, edgecolor='black')
plt.axvline(x=10, color='red', linestyle='--', label='Flag threshold at 10%')
plt.xlabel('% Rush Hour Snapshots Empty')
plt.ylabel('Number of Stations')
plt.title('Distribution of Station Empty Rates During Rush Hour')
plt.legend()
plt.show()

- The percentages under 10% are diluting the graph so I will filter for greater than 10%

In [ ]:
greater_10 = df_empty['pct_empty'][df_empty['pct_empty']>= 10 ]
print(f"precentage of stations that become empty 10% or less of the time= {round(len(greater_10) / len(df_empty['pct_empty'])* 100,2)}%" )

In [ ]:
#graphing a histogram
plt.figure(figsize=(10,6))
greater_10 = df_empty['pct_empty'][df_empty['pct_empty']>= 10 ]

plt.hist(greater_10, bins = 30, edgecolor='black')
plt.axvline(x=50, color='red', linestyle='--', label='50% threshold')
plt.axvline(x=25, color='green',linestyle='--', label = '25% threshold')
#plt.axvline(x=10, color='yellow',linestyle='--', label = '10% threshold')
plt.xlabel('% Rush Hour Snapshots Empty')
plt.ylabel('Number of Stations')
plt.title('Distribution of Stations Empty More Than 10% During Rush Hour')
plt.legend()

plt.savefig(data_dir / 'empty_distribution.png')
plt.show()

### Print percent empty distribution

In [ ]:
cursor.execute('''
SELECT 
    CASE
        WHEN pct_empty >= 50 THEN '50+'
        WHEN pct_empty >= 25 THEN '25-50'
        WHEN pct_empty >= 10 THEN '10-25%'
        WHEN pct_empty >= 5 THEN '5-10%'
        ELSE 'under 5%'
    END AS bucket,
    COUNT(*) AS station_count
FROM(
      SELECT station_id,
            COUNT(*) AS rush_hour_snapshots,
            100 * SUM(CASE WHEN num_bikes_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_empty
      FROM availability_snapshots
      WHERE EXTRACT(HOUR FROM captured_at) IN (7,8,9,17,18,19)
      GROUP BY station_id
      HAVING COUNT(*)> 100
    )
GROUP BY 1
ORDER BY 1 DESC;
''')
cursor.fetchall()

In [ ]:
cursor.execute('''

SELECT 
    s.station_id,
    COUNT(*) as trips_started
FROM rebalancing_flags rf
JOIN stations s ON rf.station_id = s.station_id
LEFT JOIN trips t ON s.short_name = t.start_station_id
WHERE rf.flag_type = 'chronic_full' AND rf.severity = 'high'
GROUP BY s.station_id
ORDER BY trips_started;
''')
cursor.fetchall()

### Print percent full distribution

In [ ]:
cursor.execute('''
WITH imbalance_tbl AS(
	SELECT 
        station_id,
		ABS(sum(avg_net_flow)) AS abs_avg_flow
	FROM hourly_patterns
	WHERE hour_of_day IN (7,8,9,17,18)
	GROUP BY station_id
    )

    SELECT 
        CASE 
           WHEN abs_avg_flow >50 THEN '50+'
           WHEN abs_avg_flow >25 THEN '25-50'
           WHEN abs_avg_flow >10 THEN '10-25'
           ELSE 'under 10'
        END AS imbalanced_bucket,
        COUNT(*)
        
      FROM imbalance_tbl 
      GROUP BY imbalanced_bucket;
    
''')
cursor.fetchall()

In [ ]:
import matplotlib.pyplot as plt

buckets = ['under 5%', '5-10%', '10-25%', '25-50%', '50%+']
empty_counts = [1540, 283, 316, 118, 43]
full_counts = [1785, 184, 225, 81, 25]

x = range(len(buckets))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([i - width/2 for i in x], empty_counts, width, label='Chronic Empty')
ax.bar([i + width/2 for i in x], full_counts, width, label='Chronic Full')

ax.set_xlabel('% of Rush Hour Snapshots')
ax.set_ylabel('Number of Stations')
ax.set_title('Station Availability Problems During Rush Hour')
ax.set_xticks(x)
ax.set_xticklabels(buckets)
ax.legend()

plt.tight_layout()
plt.savefig('availability_distribution.png')
plt.show()

In [ ]:
query = '''
   SELECT station_id,
            COUNT(*) AS rush_hour_snapshots,
            100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_full
      FROM availability_snapshots
      WHERE EXTRACT(HOUR FROM captured_at) IN (7,8,9,17,18,19)
      GROUP BY station_id
'''
df_full = pd.read_sql(query, conn)

In [ ]:
df_full.shape
df_full.head()

In [ ]:
empty_10 = df_full['pct_full'][df_full['pct_full']> 10 ]
#graphing a histogram
plt.figure(figsize=(10,6))


plt.hist(empty_10, bins = 30, edgecolor='black')
plt.axvline(x=50, color='red', linestyle='--', label='50% threshold')
plt.axvline(x=25, color='green',linestyle='--', label = '25% threshold')
#plt.axvline(x=75, color='yellow',linestyle='--', label = '75% threshold')
plt.xlabel('% Rush Hour Snapshots Full')
plt.ylabel('Number of Stations')
plt.title('Distribution of Stations Full More Than 10% During Rush Hour')
plt.legend()

plt.savefig(data_dir / 'full_distribution.png')
plt.show()

### Look at full stations during rush hour in percent

In [ ]:
cursor.execute('''
SELECT CASE
    WHEN pct_full >= 50 THEN '50+'
    WHEN pct_full >= 25 THEN '25-50'
    WHEN pct_full >= 10 THEN '10-25%'
    WHEN pct_full >= 5 THEN '5-10%'
    ELSE 'under 5%'
    END AS bucket,
    COUNT(*) AS station_count
FROM(
      SELECT station_id,
            COUNT(*) AS rush_hour_snapshots,
            100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_full
      FROM availability_snapshots
      WHERE EXTRACT(HOUR FROM captured_at) IN (7,8,9,17,18,19)
      GROUP BY station_id
      HAVING COUNT(*)> 100
    )
GROUP BY 1
ORDER BY 1 DESC;
''')
cursor.fetchall()

#### Chronic empty

cursor.execute(
    '''
    WITH pct_empty_tbl AS (
    SELECT
        station_id,
        COUNT(*) AS rush_hour_snapshots,
        100.0 * SUM(CASE WHEN num_bikes_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_empty
    FROM
        availability_snapshots
    WHERE
        EXTRACT(HOUR FROM captured_at) IN (7, 8, 9, 17, 18)
    GROUP BY station_id
    HAVING COUNT(*) > 100)
        
INSERT INTO rebalancing_flags (station_id, flag_type, severity, detected_at, notes)
    SELECT station_id,
    	'chronic_empty',
    	CASE 
    		WHEN pct_empty >= 50 THEN 'high' 
    		WHEN pct_empty >= 25 THEN 'medium'
    		ELSE 'low'
    	END,
    	NOW(),
    	'Empty ' || ROUND(pct_empty,1) || '% of rush hour snapshots'
    FROM pct_empty_tbl
    WHERE pct_empty >= 10;
    '''
)
cursor.fetchall()

### Test before inserting chronic full flag

In [ ]:
cursor.execute('''
WITH pct_full_tbl AS (
    SELECT
        station_id,
        100.0 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_full
    FROM availability_snapshots
    WHERE EXTRACT(HOUR FROM captured_at) IN (7, 8, 9, 17, 18)
    GROUP BY station_id
    HAVING COUNT(*) > 100
    )

SELECT 
    station_id,
    'chronic_full',
    CASE 
        WHEN pct_full >= 50 THEN 'high'
        WHEN pct_full >= 25 THEN 'medium'
        ELSE 'low'
    END,
    NOW(),
    'Full ' || ROUND(pct_full::numeric, 1) || '% of rush hour snapshots'
FROM pct_full_tbl
WHERE pct_full >= 10
LIMIT 10;
'''
    )
cursor.fetchall()

### Percent full stations during rush hour

In [ ]:
cursor.execute('''
WITH imbalance_tbl AS(
	SELECT 
        station_id,
		ABS(sum(avg_net_flow)) AS abs_avg_flow
	FROM hourly_patterns
	WHERE hour_of_day IN (7,8,9,17,18)
	GROUP BY station_id
    )

    SELECT 
        CASE 
           WHEN abs_avg_flow >50 THEN '50+'
           WHEN abs_avg_flow >25 THEN '25-50'
           WHEN abs_avg_flow >10 THEN '10-25'
           ELSE 'under 10'
        END AS imbalanced_bucket,
        COUNT(*)
        
      FROM imbalance_tbl 
      GROUP BY imbalanced_bucket;
    
''')
cursor.fetchall()

In [ ]:
cursor.execute(
    '''
    WITH pct_empty_tbl AS (
    SELECT
        station_id,
        COUNT(*) AS rush_hour_snapshots,
        
        100.0 * SUM(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_empty
    FROM
        availability_snapshots
    WHERE
        EXTRACT(HOUR FROM captured_at) IN (7, 8, 9, 17, 18)
    GROUP BY station_id
    HAVING COUNT(*) > 100)
        

    SELECT station_id,
    	'chronic_empty',
    	CASE 
    		WHEN pct_empty >= 50 THEN 'high' 
    		WHEN pct_empty >= 25 THEN 'medium'
    		ELSE 'low'
    	END,
    	NOW(),
    	'Empty ' || ROUND(pct_empty::NUMERIC,1) || '% of rush hour snapshots'
    FROM pct_empty_tbl;
    '''
)
cursor.fetchall()

### to see if there are stations in common between empty and high imbalance


In [ ]:
cursor.execute(
    '''	
    WITH imbalanced_tbl AS(
        SELECT 
            station_id
    	FROM hourly_patterns
    	WHERE hour_of_day IN (7,8,9,17,18)
    	GROUP BY station_id 
        HAVING ABS(SUM(avg_net_flow)) >= 25
    )
    SELECT COUNT(*)   
    FROM imbalanced_tbl it
    JOIN rebalancing_flags rf ON it.station_id = rf.station_id
    AND rf.flag_type ='chronic empty'
    limit 20;
     
    '''
    
)
cursor.fetchall()

### Test before inserting

In [ ]:
cursor.execute(
    '''	
    WITH imbalance_tbl AS(
	SELECT 
        station_id,
        SUM(avg_net_flow) AS total_flow
	FROM hourly_patterns
	WHERE hour_of_day IN (7,8,9,17,18)
	GROUP BY station_id
)

SELECT station_id,
    'high imbalance',
    CASE 
        WHEN ABS(total_flow) >= 50 THEN 'high'
        ELSE 'medium'
    END,
    NOW(),
    CASE 
        WHEN total_flow < 0 THEN 'Draining ' || ABS(total_flow)  || ' bikes during rush period'
        ELSE 'Accumulating ' || total_flow  || ' bikes during rush period'
    END
FROM imbalance_tbl
WHERE ABS(total_flow) > 25
LIMIT 10;
''')
cursor.fetchall()

In [ ]:
cursor.execute('select count(*) from rebalancing_flags')
cursor.fetchone()

In [ ]:
conn.close()